In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

# ============================================================
# 1. Inputs and outputs
# ============================================================

EXPOSURE_DIR = Path("../outputs/population_exposure")
CLIMATE_DIR = Path("../outputs/climate")

OUT_BASE = Path(
    "../outputs/panel_exposure_climate"
)

SCHEMES = [
    "local_p90",
    "local_p95",
    "city_p90",
    "city_p95",
    "fixed_tx35_tn25",
]

KEY = ["city", "cell_id", "year"]

MAIN = [
    "t2m_mean_C",
    "rh_mean_pct",
    "tp_annual_mm",
]

CLIMATE_VARS = MAIN + [
    "d2m_mean_C",
    "skt_mean_C",
    "sp_mean_hPa",
]


# ============================================================
# 2. Utility functions
# ============================================================

def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


def as_bool(s):
    z = s.astype(str).str.lower()

    require(
        z.isin(["true", "false", "1", "0"]).all(),
        f"Abnormal boolean column: {s.name}",
    )

    return z.isin(["true", "1"])


def file_info(path):
    s = path.stat()

    return {
        "path": str(path),
        "bytes": s.st_size,
        "mtime_ns": s.st_mtime_ns,
        "sha256": hashlib.sha256(
            path.read_bytes()
        ).hexdigest(),
    }


def validate_panel_keys(df, label):
    require(
        not df[KEY].isna().any().any(),
        f"{label}: missing keys.",
    )

    require(
        not df.duplicated(KEY).any(),
        f"{label}: duplicate keys.",
    )

    require(
        len(df) == 6520 and df.city.nunique() == 35,
        f"{label}: not 35 cities and 6520 rows.",
    )

    require(
        df[["city", "cell_id"]]
        .drop_duplicates()
        .shape[0] == 652,
        f"{label}: not 652 grid cells.",
    )

    require(
        set(df.year) == set(range(2016, 2026)),
        f"{label}: years do not match.",
    )

    require(
        df.groupby(["city", "cell_id"])
        .size()
        .eq(10)
        .all(),
        f"{label}: incomplete years for some grid cells.",
    )


def merge_one(base, climate):
    require(
        not base.duplicated(KEY).any()
        and not climate.duplicated(KEY).any(),
        "Duplicate input keys.",
    )

    overlap = (
        set(base.columns) & set(climate.columns)
    ) - set(KEY)

    require(
        not overlap,
        "Non-key fields with the same name found, stopping to avoid silent overwriting: "
        f"{sorted(overlap)}",
    )

    base = base.sort_values(KEY).reset_index(drop=True)

    result = base.merge(
        climate,
        on=KEY,
        how="outer",
        validate="one_to_one",
        indicator=True,
    )

    require(
        result._merge.eq("both").all(),
        "Keys of the population exposure and climate results do not fully match.",
    )

    result = (
        result.drop(columns="_merge")
        .sort_values(KEY)
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        result[base.columns],
        base,
        check_exact=True,
    )

    heat_ok = as_bool(result.annual_complete)

    require(
        heat_ok.equals(
            as_bool(result.heatwave_annual_complete)
        ),
        "Heatwave validity flags differ between the two inputs.",
    )

    main_ok = result[MAIN].notna().all(axis=1)
    all_ok = result[CLIMATE_VARS].notna().all(axis=1)

    require(
        main_ok.equals(
            as_bool(result.climate_main_complete)
        ),
        "Main climate completeness flags differ.",
    )

    require(
        all_ok.equals(
            as_bool(result.climate_all_complete)
        ),
        "All-climate completeness flags differ.",
    )

    result["climate_missing_variables"] = (
        result[CLIMATE_VARS]
        .isna()
        .apply(
            lambda r: "|".join(
                r.index[r.to_numpy()]
            ),
            axis=1,
        )
    )

    result["climate_imputed"] = False

    result["heatwave_and_main_climate_complete"] = (
        heat_ok & main_ok
    )

    result["climate_missing_reason"] = np.where(
        all_ok,
        "none",
        "source_missing_at_nearest_grid_cell",
    )

    return result


# ============================================================
# 3. Merging, output and validation
# ============================================================

def run_merge():
    climate_file = (
        CLIMATE_DIR / "climate_annual_2016_2025.csv"
    )

    climate_summary = CLIMATE_DIR / "summary.json"

    inputs = [climate_file, climate_summary] + [
        EXPOSURE_DIR / f"exposure_{s}_2016_2025.csv"
        for s in SCHEMES
    ]

    for p in inputs:
        require(p.is_file(), f"Input file not found: {p}")

    climate_report = json.loads(
        climate_summary.read_text(encoding="utf-8")
    )

    require(
        climate_report["status"] == "completed",
        "Climate computation not completed.",
    )

    provenance = [file_info(p) for p in inputs]

    climate = pd.read_csv(climate_file)
    validate_panel_keys(climate, "climate results")

    require(
        climate.loc[
            climate.tp_valid_months.lt(12),
            "tp_annual_mm",
        ].isna().all(),
        "Records with missing precipitation months show a non-missing annual total.",
    )

    out = OUT_BASE

    out.mkdir(parents=True, exist_ok=True)
    (out / "FAILED.txt").unlink(missing_ok=True)

    print("Output folder:", out, flush=True)

    rows = []
    output_files = []

    try:
        for scheme in SCHEMES:
            base = pd.read_csv(
                EXPOSURE_DIR
                / f"exposure_{scheme}_2016_2025.csv"
            )

            validate_panel_keys(base, scheme)

            require(
                base.scheme.eq(scheme).all(),
                f"{scheme}: scheme labels differ.",
            )

            merged = merge_one(base, climate)

            paper = merged.loc[
                merged.year.between(2016, 2023)
            ].copy()

            require(
                len(paper) == 5216,
                "Paper-period row count changed.",
            )

            k = ~paper.heatwave_and_main_climate_complete & as_bool(
                paper.heatwave_annual_complete
            )

            # ------------------------------------------------
            # Save the panel of this scheme
            # ------------------------------------------------
            path = (
                out / f"panel_{scheme}_2016_2023.csv"
            )

            paper.to_csv(
                path,
                index=False,
                encoding="utf-8-sig",
            )

            saved = pd.read_csv(
                path,
                keep_default_na=True,
            )

            saved["climate_missing_variables"] = (
                saved.climate_missing_variables
                .fillna("")
            )

            pd.testing.assert_frame_equal(
                saved,
                paper.reset_index(drop=True),
                check_dtype=False,
                check_exact=False,
                rtol=1e-12,
                atol=1e-10,
            )

            output_files.append({
                "file": path.name,
                "rows": len(paper),
            })

            rows.append({
                "scheme": scheme,
                "rows_2016_2023": len(paper),
                "cities": paper.city.nunique(),
                "city_years": len(
                    paper[["city", "year"]]
                    .drop_duplicates()
                ),
                "heatwave_complete_rows": int(
                    as_bool(paper.annual_complete).sum()
                ),
                "heatwave_and_main_climate_complete_rows": int(
                    paper.heatwave_and_main_climate_complete.sum()
                ),
                "main_climate_missing_rows": int(
                    (
                        ~as_bool(
                            paper.climate_main_complete
                        )
                    ).sum()
                ),
                "heatwave_valid_climate_missing_rows": int(k.sum()),
                "dropped_rows": 0,
                "imputed_rows": 0,
            })

            if scheme == "local_p90":
                columns = (
                    KEY
                    + ["lat", "lon"]
                    + CLIMATE_VARS
                    + [
                        "heatwave_annual_complete",
                        "climate_main_complete",
                        "climate_missing_variables",
                        "climate_missing_reason",
                    ]
                )

                paper.loc[
                    ~as_bool(paper.climate_all_complete),
                    columns,
                ].to_csv(
                    out
                    / "climate_missing_records_2016_2023.csv",
                    index=False,
                    encoding="utf-8-sig",
                )

                paper.loc[k, columns].to_csv(
                    out
                    / "heatwave_valid_climate_missing_2016_2023.csv",
                    index=False,
                    encoding="utf-8-sig",
                )

            print(
                f"[{scheme}] study period {len(paper)} rows, "
                f"heatwave-valid rows without main climate variables: {int(k.sum())}; "
                "validation passed.",
                flush=True,
            )

        # ----------------------------------------------------
        # Confirm that the inputs did not change during the run
        # ----------------------------------------------------
        for info in provenance:
            require(
                file_info(Path(info["path"])) == info,
                "Input files changed during the run.",
            )

        summary_table = pd.DataFrame(rows)

        summary_table.to_csv(
            out / "merge_checks_by_scheme.csv",
            index=False,
            encoding="utf-8-sig",
        )

        pd.DataFrame(provenance).to_csv(
            out / "input_manifest.csv",
            index=False,
            encoding="utf-8-sig",
        )

        report = {
            "status": "completed",
            "stage": (
                "population_exposure_climate_intermediate_panel"
            ),
            "join_keys": KEY,
            "source_files": provenance,
            "outputs": output_files,
            "checks": rows,
        }

        (out / "merge_summary.json").write_text(
            json.dumps(
                report,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

        print("Output folder:", out)

        return out, summary_table

    except (Exception, KeyboardInterrupt) as exc:
        (out / "FAILED.txt").write_text(
            repr(exc),
            encoding="utf-8",
        )
        raise


# ============================================================
# 4. Execution
# ============================================================

NEW_PANEL_DIR, PANEL_MERGE_CHECKS = run_merge()